# Multall Theory

# Chapter 01: Power Generation

Meangen

In [ ]:
"""Wrapper function to create meangen.in and run MEANGEN."""
from pathlib import Path
import subprocess
import shutil


def run_meangen(
    meangen_executable,
    meangen_input_text=None,
    meangen_input_file=None,
    run_dir='.',
    timeout_seconds=120,
    output_file='stagen.dat',
):
    """Create meangen.in in run_dir, run MEANGEN, and return run metadata.

    Provide exactly one of:
    - meangen_input_text: raw text content for meangen.in
    - meangen_input_file: path to an existing input file
    """
    if meangen_input_text is None and meangen_input_file is None:
        raise ValueError('Provide meangen_input_text or meangen_input_file.')
    if meangen_input_text is not None and meangen_input_file is not None:
        raise ValueError('Provide only one of meangen_input_text or meangen_input_file.')

    run_path = Path(run_dir).expanduser().resolve()
    exe_path = Path(meangen_executable).expanduser().resolve()

    if not exe_path.is_file():
        raise FileNotFoundError(f'MEANGEN executable not found: {exe_path}')

    run_path.mkdir(parents=True, exist_ok=True)

    meangen_in_path = run_path / 'meangen.in'
    if meangen_input_text is not None:
        meangen_in_path.write_text(meangen_input_text, encoding='utf-8')
    else:
        input_path = Path(meangen_input_file).expanduser().resolve()
        if not input_path.is_file():
            raise FileNotFoundError(f'MEANGEN input file not found: {input_path}')
        shutil.copyfile(input_path, meangen_in_path)

    result = subprocess.run(
        [str(exe_path)],
        cwd=str(run_path),
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
        check=False,
    )

    output_path = run_path / output_file
    success = result.returncode == 0 and output_path.exists()

    return {
        'success': success,
        'returncode': result.returncode,
        'stdout': result.stdout,
        'stderr': result.stderr,
        'run_dir': str(run_path),
        'meangen_in': str(meangen_in_path),
        'output_file': str(output_path),
        'output_exists': output_path.exists(),
    }

In [ ]:
"""Example Cell: turbine case using the MEANGEN wrapper."""
from pathlib import Path

# Update this to your compiled MEANGEN executable path.
# Example: Path('multallopenDURHAM_V2A/code/meangen-17.4.x')
meangen_exe = Path('multallopenDURHAM_V2A/code/meangen-17.4.x')

# Turbine sample input shipped with this repo.
sample_turbine_input = Path(
    'multall-open/MEANGEN/sample-meangen.in-data-sets/meangen-RI-turbine.in'
 )

# Working directory for generated files (meangen.in, stagen.dat, etc.).
run_dir = Path('runs/turbine_case_01')

if not meangen_exe.exists():
    print('MEANGEN executable not found. Compile meangen first, then re-run this cell.')
    print(f'Expected path: {meangen_exe.resolve()}')
else:
    meangen_result = run_meangen(
        meangen_executable=meangen_exe,
        meangen_input_file=sample_turbine_input,
        run_dir=run_dir,
        timeout_seconds=180,
        output_file='stagen.dat',
    )

    print('Run success:', meangen_result['success'])
    print('Return code:', meangen_result['returncode'])
    print('Output exists:', meangen_result['output_exists'])
    print('stagen.dat path:', meangen_result['output_file'])

    # Print tail of stdout to quickly inspect run status.
    stdout_lines = meangen_result['stdout'].splitlines()
    print('\n--- MEANGEN stdout tail ---')
    for line in stdout_lines[-25:]:
        print(line)

    if meangen_result['stderr'].strip():
        print('\n--- MEANGEN stderr ---')
        print(meangen_result['stderr'])

In [ ]:
"""Visualize MEANGEN results from stagen.dat (static pressures through row)."""
from pathlib import Path
import re
import matplotlib.pyplot as plt


def extract_static_pressure_rows(stagen_path):
    """Parse lines containing 'STATIC PRESSURES THROUGH ROW' and return 5-value arrays."""
    rows = []
    label = 'STATIC PRESSURES THROUGH ROW'
    number_pattern = re.compile(r'[-+]?\d*\.?\d+(?:[Ee][-+]?\d+)?')

    with open(stagen_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if label in line.upper():
                nums = [float(x) for x in number_pattern.findall(line)]
                if len(nums) >= 5:
                    rows.append(nums[:5])

    return rows


stagen_file = Path('runs/turbine_case_01/stagen.dat')

if not stagen_file.exists():
    print(f'No stagen.dat found at: {stagen_file.resolve()}')
    print('Run the Example Cell first.')
else:
    pressure_rows = extract_static_pressure_rows(stagen_file)

    if not pressure_rows:
        print('No "STATIC PRESSURES THROUGH ROW" entries found in stagen.dat.')
    else:
        x_labels = ['RPM', 'P1', 'P2', 'P3', 'P4']
        plt.figure(figsize=(10, 5))

        for i, row in enumerate(pressure_rows, start=1):
            plt.plot(x_labels, row, marker='o', linewidth=1.8, label=f'Row {i}')

        plt.title('MEANGEN Output: Static Pressures Through Row')
        plt.ylabel('Value (MEANGEN units)')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

        print(f'Parsed {len(pressure_rows)} row(s) from {stagen_file}')

Stagen

Multall